# Lab 0 — CAD & Visualization Workspace

**Ambiente de suporte visual para as Labs 1 (Materiais) e 2 (Estrutural).**

Recursos:
- Geometria 3D de pá de turbina eólica
- Visualização de perfis aerodinâmicos (NACA)
- Diagramas de tensão e momento
- Envelope de falha (Tsai-Wu, Max Stress)
- Vista de laminação (stacking sequence)
- Rosa dos ventos
- Mapa de cores de tensão
- Exportação STL para CAD/impressão 3D

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../modules'))
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from cad_visualization import (
    AirfoilCoordinates, BladeGeometry, LaminateView,
    WindRose, stress_color_map, failure_envelope_points,
    beam_stress_diagram
)
print('Lab 0 — CAD workspace loaded')

In [ ]:
# --- Lab 1 Support: Material Visualization ---
lam = LaminateView(layers=[
    {'material': 'graphite_coating', 'thickness_mm': 0.5, 'angle': 0},
    {'material': 'paper_mache_core', 'thickness_mm': 4.0, 'angle': 45},
    {'material': 'paper_mache_core', 'thickness_mm': 4.0, 'angle': -45},
    {'material': 'graphite_coating', 'thickness_mm': 0.5, 'angle': 0},
])
stack = lam.stacking_data()
print(f'=== Laminate Stack ({stack["total_thickness_mm"]}mm, {stack["n_layers"]} layers) ===')
for l in stack['layers']:
    print(f'  {l["material"]:20s} {l["thickness_mm"]:4.1f}mm @ {l["angle_deg"]:3.0f}°')

# Airfoil
af = AirfoilCoordinates('NACA4412', chord_m=0.3)
x_af, y_af = af.coordinates()
print(f'\nAirfoil {af.name}: {len(x_af)} points, chord={af.chord_m}m')

In [ ]:
# --- Lab 2 Support: Blade & Structure Visualization ---
blade = BladeGeometry(length_m=1.5, root_chord_m=0.3, tip_chord_m=0.1)
sections = blade.blade_sections()
print('=== Blade Sections ===')
for s in sections:
    print(f'  r={s["r_m"]:4.2f}m chord={s["chord_m"]:4.3f}m twist={s["twist_deg"]:5.1f}°')

# Stress diagram for a simple load case
diagram = beam_stress_diagram(L_m=1.5, forces_N=[50, 30], positions=[0.5, 1.0])
max_M = max(abs(m) for m in diagram['moment_Nm'])
print(f'\nMax bending moment: {max_M:.1f} Nm')

In [ ]:
# --- Visualizations ---
fig = plt.figure(figsize=(18, 12))

# 1. Airfoil
ax = fig.add_subplot(2, 3, 1)
ax.plot(x_af, y_af, 'b-', linewidth=1.5)
ax.set_aspect('equal'); ax.set_title(f'Airfoil: {af.name}'); ax.grid(True)

# 2. Stacking sequence
ax = fig.add_subplot(2, 3, 2)
colors = ['#555555', '#D2B48C', '#D2B48C', '#555555']
for i, l in enumerate(stack['layers']):
    ax.barh(0, l['thickness_mm'], left=l['z_start_mm'],
            height=0.6, color=colors[i], edgecolor='black')
    ax.text(l['z_start_mm'] + l['thickness_mm']/2, 0,
            f"{l['angle_deg']}°", ha='center', va='center')
ax.set_xlabel('Thickness (mm)'); ax.set_title('Laminate Stack')
ax.set_yticks([]); ax.grid(True, axis='x')

# 3. Shear diagram
ax = fig.add_subplot(2, 3, 3)
ax.plot(diagram['x_m'], diagram['shear_N'], 'r-', linewidth=2)
ax.fill_between(diagram['x_m'], diagram['shear_N'], alpha=0.3)
ax.set_xlabel('Span (m)'); ax.set_ylabel('Shear (N)')
ax.set_title('Shear Diagram'); ax.grid(True)

# 4. Moment diagram
ax = fig.add_subplot(2, 3, 4)
ax.plot(diagram['x_m'], diagram['moment_Nm'], 'b-', linewidth=2)
ax.fill_between(diagram['x_m'], diagram['moment_Nm'], alpha=0.3)
ax.set_xlabel('Span (m)'); ax.set_ylabel('Moment (Nm)')
ax.set_title('Moment Diagram'); ax.grid(True)

# 5. Failure envelope
ax = fig.add_subplot(2, 3, 5)
env = failure_envelope_points('tsai_wu', Xt=50, Xc=30, Yt=15, Yc=10, S=8)
if env['sigma_11'] and env['sigma_22']:
    ax.plot(env['sigma_11'], env['sigma_22'], 'b-', linewidth=2, label='Tsai-Wu')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.fill_between(env['sigma_11'], env['sigma_22'], alpha=0.2)
ax.set_xlabel('σ₁₁ (MPa)'); ax.set_ylabel('σ₂₂ (MPa)')
ax.set_title('Failure Envelope'); ax.grid(True); ax.axis('equal')

# 6. Wind rose
ax = fig.add_subplot(2, 3, 6, projection='polar')
wr = WindRose()
data = wr.polar_data()
ax.bar(data['theta'], data['frequencies'], width=0.6, alpha=0.6)
ax.set_title('Wind Rose'); ax.set_xticklabels(wr.directions)

plt.tight_layout()
plt.show()

print('=== Lab 0 Complete: CAD & Visualization Workspace ===')
print('Drawing: airfoil, laminate stack, shear/moment diagrams')
print('Analysis: failure envelope, wind rose')